In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from shapely.geometry import Point, LineString, Polygon, MultiPoint, mapping
import numpy as np
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm
import plotly.express as px
import json

In [ ]:
# for the langoid CSV file 
df = pd.read_csv("glottolog_languoid.csv/languoid.csv") 
df.head() 

In [ ]:
df = pd.read_csv("dravidian-mother-tongues-main-n-level.csv")
df = df.dropna()
df

In [ ]:
df.columns

In [ ]:
### Nation-level Plotting

In [ ]:
### Total Mother Tongue Speakers

In [ ]:
fig, ax = plt.subplots()

# Example data
ax.barh(
    df["Mother tongue"],
    df["Total_P"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Total_P)")
ax.set_title("Number of Mother Tongue Speakers (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
# for the langoid CSV file 
df = pd.read_csv("glottolog_languoid.csv/languoid.csv") 
df.head() 

In [ ]:
df.columns

In [ ]:
tamil = df[df["name"] == "Tamil"]
tamil

In [ ]:
malayalam = df[df["name"] == "Malayalam"]
malayalam

In [ ]:
tamil_longitude_val = tamil["longitude"].iloc[0]
tamil_latitude_val = tamil["latitude"].iloc[0]
# created a geometric object in coordinate space 
tamil_point = Point(tamil_longitude_val, tamil_latitude_val)

In [ ]:
type(tamil_point)

In [ ]:
## need to put the point into a geodataframe 
tamil_gdf = gpd.GeoDataFrame(
    {
        "name": ["Tamil"],
        "iso": ["tam"],
        "geometry": [tamil_point]
    },
    crs="EPSG:4326"
)

In [ ]:
### NOW TO GET A POLYGON LAYER (INDIA) 
india_states = gpd.read_file("gadm41_IND_shp/gadm41_IND_1.shp") 
india_states.head()

## State Level Plotting

In [ ]:
### State Level Graphs
state_df = pd.read_csv("dravidian-mother-tongues-main-state-level.csv")
state_df

In [ ]:
india_states["NAME_1"].unique()

In [ ]:
india_states[["NAME_1", "geometry"]].head()

### Malayalam 

In [ ]:
malayalam_df = state_df[state_df["Mother Tongue"] == "Malayalam"]
malayalam_df

In [ ]:
## Plotting Total_P on Choropleth map

In [ ]:
## Merging Malayalam data with india_states 
mal_map = india_states.merge(malayalam_df, left_on="NAME_1", right_on="State_UT", how="left")
mal_map.head()

In [ ]:
mal_map_slim = mal_map[["NAME_1", "Mother Tongue", "Total_P"]].dropna()
mal_map_slim

In [ ]:
# The rows GADM has for Maharashtra
mal_map[mal_map["NAME_1"].str.contains("Maharashtra", case=False, na=False)][
    ["NAME_1", "State_UT", "Mother Tongue", "Total_P", "geometry"]
]

# Seeing if NAME_1 values have ANY null Total_P after the merge?
mal_map[mal_map["Total_P"].isna()]["NAME_1"].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))

mal_map.plot(
    ax=ax,
    column="Total_P",
    legend=True,
      missing_kwds={
        "color": "lightgrey",
        "edgecolor": "red",
        "hatch": "///",
        "label": "Missing values",
    },
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}
);

ax.set_title(
    "Malayalam Mother Tongue Speakers by State (C-16 Census 2011)", 
    fontsize=12
);

In [ ]:
## Not Including Kerala 

mal_map_outside = mal_map[mal_map["NAME_1"] != "Kerala"]
mal_map_outside[["NAME_1", "Mother Tongue", "Total_P"]].dropna()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))

mal_map_outside.plot(
    ax=ax,
    column="Total_P",
    legend=True,
    missing_kwds={
        "color": "lightgrey",
        "edgecolor": "red",
        "hatch": "///",
        "label": "Missing values",
    },
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}
);


## Adding Kerala to the map as a red element separate from Total_P
kerala_geom = mal_map[mal_map["NAME_1"] == "Kerala"]
kerala_geom.plot(
    ax=ax, 
    color="red", 
    edgecolor="darkred",
    linewidth=1, 
    label="Kerala (native state, excluded from scale)"
)

ax.set_title(
    "Malayalam Mother Tongue Speakers not including Kerala (C-16 Census 2011)", 
    fontsize=12
);

### Tamil 

In [ ]:
tamil_df = state_df[state_df["Mother Tongue"] == "Tamil"]
tamil_df

In [ ]:
## Plotting Total_P on Choropleth map
## Merging Tamil data with india_states 
tamil_map = india_states.merge(tamil_df, left_on="NAME_1", right_on="State_UT", how="left")
tamil_map.head()

In [ ]:
tam_map_slim = tamil_map[["NAME_1", "Mother Tongue", "Total_P"]].dropna()
tam_map_slim

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))

tamil_map.plot(
    ax=ax,
    column="Total_P",
    legend=True,
      missing_kwds={
        "color": "lightgrey",
        "edgecolor": "red",
        "hatch": "///",
        "label": "Missing values",
    },
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}
);

ax.set_title(
    "Tamil Mother Tongue Speakers by State (C-16 Census 2011)", 
    fontsize=12
);

In [ ]:
## Not including Tamil Nadu


tam_map_outside = tamil_map[tamil_map["NAME_1"] != "Tamil Nadu"]
tam_map_outside[["NAME_1", "Mother Tongue", "Total_P"]].dropna()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))

tam_map_outside.plot(
    ax=ax,
    column="Total_P",
    legend=True,
    missing_kwds={
        "color": "lightgrey",
        "edgecolor": "red",
        "hatch": "///",
        "label": "Missing values",
    },
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}
);

## Adding Tamil Nadu to the map as a red element separate from Total_P
tamil_nadu_geom = mal_map[mal_map["NAME_1"] == "Tamil Nadu"]
tamil_nadu_geom.plot(
    ax=ax, 
    color="red", 
    edgecolor="darkred",
    linewidth=1, 
    label="Tamil (native state, excluded from scale)"
)


ax.set_title(
    "Tamil Mother Tongue Speakers not including Tamil Nadu (C-16 Census 2011)", 
    fontsize=12
);

In [ ]:
### Telugu 

In [ ]:
telugu_df = state_df[state_df["Mother Tongue"] == "Telugu"]
telugu_df

In [ ]:
## Plotting Total_P on Choropleth map
## Merging Telugu data with india_states 
telugu_map = india_states.merge(telugu_df, left_on="NAME_1", right_on="State_UT", how="left")
telugu_map.head()

In [ ]:
telugu_map[["NAME_1", "Mother Tongue", "Total_P"]].dropna()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))

telugu_map.plot(
    ax=ax,
    column="Total_P",
    legend=True,
     missing_kwds={
        "color": "lightgrey",
        "edgecolor": "red",
        "hatch": "///",
        "label": "Missing values",
    },
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}
);


ax.set_title(
    "Telugu Mother Tongue Speakers by State (C-16 Census 2011)", 
    fontsize=12
);

In [ ]:
### Not including Andhra Pradesh


telugu_map_outside = telugu_map[telugu_map["NAME_1"] != "Andhra Pradesh"]
telugu_map_outside[["NAME_1", "Mother Tongue", "Total_P"]].dropna()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))

telugu_map_outside.plot(
    ax=ax,
    column="Total_P",
    legend=True,
    missing_kwds={
        "color": "lightgrey",
        "edgecolor": "red",
        "hatch": "///",
        "label": "Missing values",
    },
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}
);


## Adding Andhra Pradesh to the map as a red element separate from Total_P
andhra_geom = mal_map[mal_map["NAME_1"] == "Andhra Pradesh"]
andhra_geom.plot(
    ax=ax, 
    color="red", 
    edgecolor="darkred",
    linewidth=1, 
    label="Andhra Pradesh (native state, excluded from scale)"
)
ax.set_title(
    "Telugu Mother Tongue Speakers by State (C-16 Census 2011)", 
    fontsize=12
);

In [ ]:
### Kannada 

In [ ]:
kannada_df = state_df[state_df["Mother Tongue"] == "Kannada"]
kannada_df

In [ ]:
## Plotting Total_P on Choropleth map
## Merging Telugu data with india_states 
kannada_map = india_states.merge(kannada_df, left_on="NAME_1", right_on="State_UT", how="left")
kannada_map.head()

In [ ]:
kannada_map[["NAME_1", "Mother Tongue", "Total_P"]].dropna()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))

kannada_map.plot(
    ax=ax,
    column="Total_P",
    legend=True,
    missing_kwds={
        "color": "lightgrey",
        "edgecolor": "red",
        "hatch": "///",
        "label": "Missing values",
    },
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}
);

ax.set_title(
    "Kannada Mother Tongue Speakers by State (C-16 Census 2011)", 
    fontsize=12
);

In [ ]:
## Not including Karntaka

kannada_map_outside = kannada_map[kannada_map["NAME_1"] != "Karnataka"]
kannada_map_outside[["NAME_1", "Mother Tongue", "Total_P"]].dropna()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))

kannada_map_outside.plot(
    ax=ax,
    column="Total_P",
    legend=True,
    missing_kwds={
        "color": "lightgrey",
        "edgecolor": "red",
        "hatch": "///",
        "label": "Missing values",
    },
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}
);


## Adding Karnataka to the map as a red element separate from Total_P
karnataka_geom = mal_map[mal_map["NAME_1"] == "Karnataka"]
karnataka_geom.plot(
    ax=ax, 
    color="red", 
    edgecolor="darkred",
    linewidth=1, 
    label="Karnataka (native state, excluded from scale)"
)

ax.set_title(
    "Kannada Mother Tongue Speakers by State (C-16 Census 2011)", 
    fontsize=12
);

In [ ]:
### Making a Multiples Map Layout of Language Distributions that include the state of origin of language

In [ ]:
all_maps = [mal_map, tamil_map, kannada_map, telugu_map]

In [ ]:
titles = ["Malayalam", "Tamil", "Kannada", "Telugu"]

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12,12))

for map_data, title, subplot in zip(all_maps, titles, ax.flatten()):
    
    map_data.plot(
        ax=subplot,
        column="Total_P",
        norm=LogNorm(),
        legend=True, 
        edgecolor="white",
        linewidth=0.5,
        legend_kwds={"label": "Mother Tongue Speaker Population (2011 Census)", "orientation": "horizontal"}
    
    );
    
    subplot.set_title(title);
    subplot.axis('off')

#print(type(graph))

fig.subplots_adjust(wspace=0.08, hspace=0.1)
fig.suptitle("Mother Tongue Spatial Distribution of Major Dravidian Languages Across India (C-16 Census 2011) (Log scale)", fontsize=15);

## adding a visual gradient

#cmap = cm.viridis
#cax = ax.

# add a ScalarMappable object 
#sm = cm.ScalarMappable(cmap=cmap)
#cbar = fig.colorbar(sm, orientation="horizontal", ax=ax, pad=0.05, ticks=None);
#cbar.set_ticks([0, 1], labels=['Low Concentration', 'High Concentration'])

#colorizer = mpl.colors.(cmap='viridis')

In [ ]:
### Making a Multiples Map Layout of Language Distributions that do NOT include the state of origin of language

In [ ]:
all_outside_maps = [mal_map_outside, tam_map_outside, telugu_map_outside, kannada_map_outside]

In [ ]:
titles = ["Malayalam", "Tamil", "Telugu", "Kannada"]

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12,12))

for map_data, title, subplot in zip(all_outside_maps, titles, ax.flatten()):
    
    map_data.plot(
        ax=subplot,
        column="Total_P",
        legend=True, 
        norm=LogNorm(),
        edgecolor="white",
        linewidth=0.5,
        legend_kwds={"label": "Mother Tongue Speaker Population (2011 Census)", "orientation": "horizontal"}
    
    );
    
    subplot.set_title(title);
    subplot.axis('off')

#print(type(graph))

fig.subplots_adjust(wspace=0.08, hspace=0.1)
fig.suptitle("Mother Tongue Spatial Distribution of Major Dravidian Languages (not including major language speaking state) Across India (C-16 Census 2011)", fontsize=15);

## adding a visual gradient

#cmap = cm.viridis
#cax = ax.

# add a ScalarMappable object 
#sm = cm.ScalarMappable(cmap=cmap)
#cbar = fig.colorbar(sm, orientation="horizontal", ax=ax, pad=0.05, ticks=None);
#cbar.set_ticks([0, 1], labels=['Low Concentration', 'High Concentration'])

#colorizer = mpl.colors.(cmap='viridis')

In [ ]:
minor_tongues_df = pd.read_csv("dravidian-mother-tongues-minor-n-level.csv")
minor_tongues_df = minor_tongues_df.dropna()
minor_tongues_df

In [ ]:
### Making Bar Charts for each column 

In [ ]:
## Total_P

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df["Mother tongue"],
    minor_tongues_df["Total_P"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Total_P)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
## Total_M

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df["Mother tongue"],
    minor_tongues_df["Total_M"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Total_M)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
## Total_F

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df["Mother tongue"],
    minor_tongues_df["Total_F"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Total_F)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
### Rural_P

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df["Mother tongue"],
    minor_tongues_df["Rural_P"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Rural_P)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
### Rural_M

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df["Mother tongue"],
    minor_tongues_df["Rural_M"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Rural_P)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
### Rural_F

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df["Mother tongue"],
    minor_tongues_df["Rural_F"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Rural_P)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
### Urban Population 

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df["Mother tongue"],
    minor_tongues_df["Urban_P"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Urban_P)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
## Urban_M 

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df["Mother tongue"],
    minor_tongues_df["Urban_M"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Urban_M)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
## Urban_F

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df["Mother tongue"],
    minor_tongues_df["Urban_F"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Urban_F)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df["Mother tongue"],
    minor_tongues_df["Total_F"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Total_F)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

# Splitting the longer bars from the shorter bars 

## Longer Bars

In [ ]:
## longer bars 
minor_tongues_df_top = minor_tongues_df.iloc[:5]
minor_tongues_df_top

### Total_P

In [ ]:
## Total_P

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_top["Mother tongue"],
    minor_tongues_df_top["Total_P"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Total_P)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

### Total_M

In [ ]:
## Total_M

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_top["Mother tongue"],
    minor_tongues_df_top["Total_M"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Total_M)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

### Total_F

In [ ]:
## Total_F

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_top["Mother tongue"],
    minor_tongues_df_top["Total_F"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Total_F)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

### Rural_P

In [ ]:
## Rural_P

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_top["Mother tongue"],
    minor_tongues_df_top["Rural_P"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Rural_P)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

### Rural_M

In [ ]:
## Rural_M

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_top["Mother tongue"],
    minor_tongues_df_top["Rural_M"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Rural_M)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

### Rural_F

In [ ]:
## Rural_F

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_top["Mother tongue"],
    minor_tongues_df_top["Rural_F"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Rural_F)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

### Urban_P

In [ ]:
## Urban_P

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_top["Mother tongue"],
    minor_tongues_df_top["Urban_P"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Urban_P)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
### Urban_M 

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_top["Mother tongue"],
    minor_tongues_df_top["Urban_M"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Urban_M)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
### Urban_F

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_top["Mother tongue"],
    minor_tongues_df_top["Urban_F"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Urban_F)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

## Shorter Bars

In [ ]:
### shorter bars 
minor_tongues_df_bottom = minor_tongues_df.iloc[-4:]
minor_tongues_df_bottom

In [ ]:
### Total_P

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_bottom["Mother tongue"],
    minor_tongues_df_bottom["Total_P"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Total_P)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
### Total_M

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_bottom["Mother tongue"],
    minor_tongues_df_bottom["Total_M"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Total_M)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
### Total_F

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_bottom["Mother tongue"],
    minor_tongues_df_bottom["Total_F"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Total_F)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
### Rural_P

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_bottom["Mother tongue"],
    minor_tongues_df_bottom["Rural_P"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Rural_P)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
### Rural_M

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_bottom["Mother tongue"],
    minor_tongues_df_bottom["Rural_M"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Rural_M)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
### Rural_F

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_bottom["Mother tongue"],
    minor_tongues_df_bottom["Rural_F"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Rural_F)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
### Urban_P

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_bottom["Mother tongue"],
    minor_tongues_df_bottom["Urban_P"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Urban_P)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
### Urban_M

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_bottom["Mother tongue"],
    minor_tongues_df_bottom["Urban_M"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Urban_M)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

In [ ]:
### Urban_F

fig, ax = plt.subplots()

# Example data
ax.barh(
    minor_tongues_df_bottom["Mother tongue"],
    minor_tongues_df_bottom["Urban_F"]
)
ax.set_ylabel("Mother Tongue")
ax.set_xlabel("Speaker Count (Urban_F)")
ax.set_title("Number of Mother Tongue Speakers for Minor Dravidian Languages (C-16)")
ax.invert_yaxis()

plt.show()

# Making a Multiples Map of Longer vs Shorter Bars 

## Multiples Map (Longer Bars) 

In [ ]:
columns = ["Total_P", "Total_M", "Total_F", "Rural_P", "Rural_M", "Rural_F", "Urban_P", "Urban_M", "Urban_F"]

In [ ]:
### Longer

fig, ax= plt.subplots(3,3, figsize=(15,12))

for position, total_x in enumerate(columns): 
    row = position//3
    col = position%3
            
    # Example data
    ax[row, col].barh(
        minor_tongues_df_top["Mother tongue"],
        minor_tongues_df_top[total_x]
    )
    ax[row, col].set_ylabel("Mother Tongue")
    ax[row, col].set_xlabel("Speaker Count")
    ax[row, col].set_title(total_x)
    ax[row, col].invert_yaxis()

fig.suptitle("Multiples Map of Other Dravidian Languages (Higher Speaker Counts) (C-16)", fontsize=15);

plt.tight_layout()
plt.show()

## Multiples Map (Shorter Bars) 

In [ ]:
### Longer

fig, ax= plt.subplots(3,3, figsize=(15,12))

for position, total_x in enumerate(columns): 
    row = position//3
    col = position%3
            
    # Example data
    ax[row, col].barh(
        minor_tongues_df_bottom["Mother tongue"],
        minor_tongues_df_bottom[total_x]
    )
    ax[row, col].set_ylabel("Mother Tongue")
    ax[row, col].set_xlabel("Speaker Count")
    ax[row, col].set_title(total_x)
    ax[row, col].invert_yaxis()


fig.suptitle("Multiples Map of Other Dravidian Languages (Lower Speaker Counts) (C-16)", fontsize=15);
plt.tight_layout()

plt.show()

In [ ]:
### Need to make this a 100% Stacked Bar Chart - working with minor_tongues_df

In [ ]:
rural_urban_df = minor_tongues_df[["Mother tongue", "Rural_P", "Urban_P", "Total_P"]].copy()
rural_urban_df

In [ ]:
## adding a new column titled rural-share and urban-share
rural_urban_df["rural-share"] = rural_urban_df["Rural_P"]/rural_urban_df["Total_P"]
rural_urban_df["urban-share"] = rural_urban_df["Urban_P"]/rural_urban_df["Total_P"]
rural_urban_df

In [ ]:
ax = rural_urban_df.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6), 
    x="Mother tongue",
    y=["rural-share", "urban-share"],
)
plt.xlabel("Mother tongue")
plt.ylabel("Share of Speakers");
plt.title("Rural & Urban Distribution of Other Dravidian Mother Tongue speakers (100% Stacked Bar Chart) C-16");

for position, population in enumerate(rural_urban_df["Total_P"]): 
    if population >= 1000000:
        population = population/1000000
        label = f"{population:.1f}m"
    elif population >= 1000: 
        population = population/1000
        label = f"{population:.1f}k"

    ax.text(position, 1.02, label, ha="center", va="bottom")
    ax.set_ylim(0, 1.15)

### kurukh/oraon and kodagu/coorgi have identical counts, i believe that was a typo needing correction

In [ ]:
## Making another stacked bar chart for male vs female speakers

In [ ]:
male_female_df = minor_tongues_df[["Mother tongue", "Total_M", "Total_F", "Total_P"]].copy()
male_female_df

In [ ]:
## adding a new column titled male-share and female-share
male_female_df["male_share"] = male_female_df["Total_M"]/male_female_df["Total_P"]
male_female_df["female-share"] = male_female_df["Total_F"]/male_female_df["Total_P"]
male_female_df

In [ ]:
ax = male_female_df.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6), 
    x="Mother tongue",
    y=["male_share", "female-share"],
)
plt.xlabel("Mother tongue")
plt.ylabel("Share of Speakers");
plt.title("Male & Female Distribution of Other Dravidian Mother Tongue speakers (100% Stacked Bar Chart) C-16");


for position, population in enumerate(male_female_df["Total_P"]): 
    if population >= 1000000:
        population = population/1000000
        label = f"{population:.1f}m"
    elif population >= 1000: 
        population = population/1000
        label = f"{population:.1f}k"

    ax.text(position, 1.02, label, ha="center", va="bottom")
ax.set_ylim(0, 1.15);

In [ ]:
## Making a stacked bar chart for the main 4 dravidian languages as well? 

In [ ]:
main_df = pd.read_csv("dravidian-mother-tongues-main-n-level.csv")
main_df = main_df.dropna()
main_df

In [ ]:
## adding a new column titled rural-share and urban-share
main_rural_urban_df = main_df[["Mother tongue", "Rural_P", "Urban_P", "Total_P"]].copy()
main_rural_urban_df

In [ ]:
## adding a new column titled rural-share and urban-share
main_rural_urban_df["rural-share"] = main_rural_urban_df["Rural_P"]/main_rural_urban_df["Total_P"]
main_rural_urban_df["urban-share"] = main_rural_urban_df["Urban_P"]/main_rural_urban_df["Total_P"]
main_rural_urban_df

In [ ]:
ax = main_rural_urban_df.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6), 
    x="Mother tongue",
    y=["rural-share", "urban-share"],
)
plt.xlabel("Mother tongue")
plt.ylabel("Share of Speakers");
plt.title("Rural & Urban Distribution of Main Dravidian Mother Tongue speakers (100% Stacked Bar Chart) C-16");

for position, population in enumerate(main_rural_urban_df["Total_P"]): 
    if population >= 1000000:
        population = population/1000000
        label = f"{population:.1f}m"
    elif population >= 1000: 
        population = population/1000
        label = f"{population:.1f}k"

    ax.text(position, 1.02, label, ha="center", va="bottom")
    ax.legend(bbox_to_anchor=(1.2, 1))
    ax.set_ylim(0, 1.15);
    

In [ ]:
## doing the same thing for male-female 
main_male_female_df = main_df[["Mother tongue", "Total_M", "Total_F", "Total_P"]].copy()
main_male_female_df

In [ ]:
## adding a new column titled male-share and female-share
main_male_female_df["male-share"] = main_male_female_df["Total_M"]/main_male_female_df["Total_P"]
main_male_female_df["female-share"] = main_male_female_df["Total_F"]/main_male_female_df["Total_P"]
main_male_female_df

In [ ]:
ax = main_male_female_df.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6), 
    x="Mother tongue",
    y=["male-share", "female-share"],
)
plt.xlabel("Mother tongue")
plt.ylabel("Share of Speakers");
plt.title("Male & Female Distribution of Main Dravidian Mother Tongue speakers (100% Stacked Bar Chart) C-16");

for position, population in enumerate(main_rural_urban_df["Total_P"]): 
    if population >= 1000000:
        population = population/1000000
        label = f"{population:.1f}m"
    elif population >= 1000: 
        population = population/1000
        label = f"{population:.1f}k"

    ax.text(position, 1.02, label, ha="center", va="bottom")
    ax.legend(bbox_to_anchor=(1.2, 1))
    ax.set_ylim(0, 1.15);

In [ ]:
### Back to mapping minority Dravidian languages? 
## Starting with Kerala again 
fig, ax = plt.subplots(figsize=(8,8))

mal_map_outside.plot(
    ax=ax,
    column="Total_P",
    legend=True,
    missing_kwds={
        "color": "lightgrey",
        "edgecolor": "red",
        "hatch": "///",
        "label": "Missing values",
    },
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}
);


## Adding Kerala to the map as a red element separate from Total_P
kerala_geom = mal_map[mal_map["NAME_1"] == "Kerala"]
kerala_geom.plot(
    ax=ax, 
    color="red", 
    edgecolor="darkred",
    linewidth=1, 
    label="Kerala (native state, excluded from scale)"
)

ax.set_title(
    "Malayalam Mother Tongue Total Speakers not including Kerala (C-16 Census 2011)", 
    fontsize=12
);

In [ ]:
## Rural 
fig, ax = plt.subplots(figsize=(8,8))

mal_map_outside.plot(
    ax=ax,
    column="Rural_P",
    legend=True,
    missing_kwds={
        "color": "lightgrey",
        "edgecolor": "red",
        "hatch": "///",
        "label": "Missing values",
    },
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}
);


## Adding Kerala to the map as a red element separate from Total_P
kerala_geom = mal_map[mal_map["NAME_1"] == "Kerala"]
kerala_geom.plot(
    ax=ax, 
    color="red", 
    edgecolor="darkred",
    linewidth=1, 
    label="Kerala (native state, excluded from scale)"
)

ax.set_title(
    "Malayalam Mother Tongue Rural Speakers not including Kerala (C-16 Census 2011)", 
    fontsize=12
);

### Multiples Map Layout

In [ ]:
## Urban
fig, ax = plt.subplots(figsize=(8,8))

mal_map_outside.plot(
    ax=ax,
    column="Urban_P",
    legend=True,
    missing_kwds={
        "color": "lightgrey",
        "edgecolor": "red",
        "hatch": "///",
        "label": "Missing values",
    },
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}
);


## Adding Kerala to the map as a red element separate from Total_P
kerala_geom = mal_map[mal_map["NAME_1"] == "Kerala"]
kerala_geom.plot(
    ax=ax, 
    color="red", 
    edgecolor="darkred",
    linewidth=1, 
    label="Kerala (native state, excluded from scale)"
)

ax.set_title(
    "Malayalam Mother Tongue Urban Speakers not including Kerala (C-16 Census 2011)", 
    fontsize=12
);

In [ ]:
### Wanting to make a Multiples Map of this
fig, ax = plt.subplots(1, 2, figsize=(14,7))

## State Maps

### Kerala 

In [ ]:
state_df = pd.read_csv("dravidian-mother-tongues-main-s-level.csv") 
state_df

In [ ]:
kerala_df = state_df[state_df["State"] == "Kerala"]
kerala_df

In [ ]:
kerala_shape = india_states[india_states["NAME_1"] == "Kerala"]
kerala_shape.plot();

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12,12))
axes = axes.flatten()

mother_tongues = kerala_df["Mother tongue"]
total_p = kerala_df["Total_P"]

for i, mother_tongue in enumerate(mother_tongues): 
    language_row = kerala_df[kerala_df["Mother tongue"] == mother_tongue]
    speaker_count = language_row["Total_P"].iloc[0]
    print(mother_tongue, speaker_count)
    ## Attaching to the map 
    language_map = kerala_shape.copy() 
    language_map["Total_P"] = speaker_count
    language_map.plot(ax=axes[i], column="Total_P", cmap="viridis", vmin=total_p.min(), vmax=total_p.max())

In [ ]:
tamilnadu_shape = india_states[india_states["NAME_1"] == "Tamil Nadu"]
tamilnadu_shape.plot();

In [ ]:
tamil_nadu_df = state_df[state_df["State"] == "Tamil Nadu"]
tamil_nadu_df

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12,12))
axes = axes.flatten()

mother_tongues = tamil_nadu_df["Mother tongue"]
total_p = tamil_nadu_df["Total_P"]

for i, mother_tongue in enumerate(mother_tongues): 
    language_row = tamil_nadu_df[tamil_nadu_df["Mother tongue"] == mother_tongue]
    speaker_count = language_row["Total_P"].iloc[0]
    print(mother_tongue, speaker_count)
    ## Attaching to the map 
    language_map = tamilnadu_shape.copy() 
    language_map["Total_P"] = speaker_count
    language_map.plot(ax=axes[i], column="Total_P", cmap="viridis", vmin=total_p.min(), vmax=total_p.max())

In [ ]:
karnataka_shape = india_states[india_states["NAME_1"] == "Karnataka"]
karnataka_shape.plot();

In [ ]:
karnataka_df = state_df[state_df["State"] == "Karnataka"]
karnataka_df

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12,12))
axes = axes.flatten()

mother_tongues = karnataka_df["Mother tongue"]
total_p = karnataka_df["Total_P"]

for i, mother_tongue in enumerate(mother_tongues): 
    language_row = karnataka_df[karnataka_df["Mother tongue"] == mother_tongue]
    speaker_count = language_row["Total_P"].iloc[0]
    print(mother_tongue, speaker_count)
    ## Attaching to the map 
    language_map = karnataka_shape.copy() 
    language_map["Total_P"] = speaker_count
    language_map.plot(ax=axes[i], column="Total_P", cmap="viridis", vmin=total_p.min(), vmax=total_p.max())

In [ ]:
andhra_shape = india_states[india_states["NAME_1"] == "Andhra Pradesh"]
andhra_shape.plot();

In [ ]:
andhra_df = state_df[state_df["State"] == "Andhra Pradesh"]
andhra_df

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12,12))
axes = axes.flatten()

mother_tongues = andhra_df["Mother tongue"]
total_p = andhra_df["Total_P"]

for i, mother_tongue in enumerate(mother_tongues): 
    language_row = andhra_df[andhra_df["Mother tongue"] == mother_tongue]
    speaker_count = language_row["Total_P"].iloc[0]
    print(mother_tongue, speaker_count)
    ## Attaching to the map 
    language_map = andhra_shape.copy() 
    language_map["Total_P"] = speaker_count
    language_map.plot(ax=axes[i], column="Total_P", cmap="viridis", vmin=total_p.min(), vmax=total_p.max())

In [ ]:
df = pd.read_csv("glottolog_languoid.csv/languoid.csv") 
df.head() 

In [ ]:
tulu = df[df["name"] == "Tulu"]
tulu

In [ ]:
gondi = df[df["name"] == "Gondi"]
gondi

# Mapping at the District Level

### Kerala at the District level

In [ ]:
district_kerala_df = pd.read_csv("dravidian_languages_kerala_district-level.csv")
district_kerala_df

In [ ]:
kerala_shape = india_states[india_states["NAME_1"] == "Kerala"]
kerala_shape.plot();

In [ ]:
districts = gpd.read_file("gadm41_IND_2.json")

In [ ]:
districts.head()

In [ ]:
districts["NAME_1"].unique()

In [ ]:
## want to see data that is about Kerala districts
districts[districts["NAME_1"] == "Kerala"]

In [ ]:
kasaragod = districts[(districts["NAME_1"] == "Kerala") & (districts["NAME_2"]=="Kasaragod")]
kasaragod

In [ ]:
# plotted kasaragod district in kerala (top-most district)
kasaragod.plot()

In [ ]:
df_kasara = district_kerala_df[district_kerala_df["District"] == "Kasaragod"]
df_kasara

In [ ]:
speaker_count = df_kasara["Total_P"].iloc[0]
kasaragod_map = kasaragod.copy()
kasaragod_map["Total_P"] = speaker_count 

In [ ]:
kasaragod_map.plot(column="Total_P", cmap="viridis") 

In [ ]:
## getting all kerala district boundaries
kerala_districts = districts[districts["NAME_1"] == "Kerala"].copy()


# taking all the malayalam rows 
malayalam_df = district_kerala_df[district_kerala_df["Mother tongue"] == "Malayalam"]

# merging 
kerala_malayalam = kerala_districts.merge(
    malayalam_df[["District", "Total_P", "Rural_P", "Urban_P"]],
    left_on = "NAME_2",
    right_on = "District", 
    how = "left", 
    validate="one_to_one"
)

In [ ]:
## making sure total_p shows for all districts once merged
kerala_malayalam 

# Malayalam Mother Tongue Speakers in Kerala

In [ ]:
## making a full district map 

fig, ax = plt.subplots(figsize=(8,8))

kerala_malayalam.plot(
    ax=ax,
    column="Total_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2001 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Malayalam Total Population Speaker Distribution Across Kerala Districts (C-16)", fontsize=12
);

In [ ]:
## making a full district map 

fig, ax = plt.subplots(figsize=(8,8))

kerala_malayalam.plot(
    ax=ax,
    column="Rural_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2001 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Malayalam Rural Population Speaker Distribution Across Kerala Districts (C-16)", fontsize=12
);

In [ ]:
## making a full district map 

fig, ax = plt.subplots(figsize=(8,8))

kerala_malayalam.plot(
    ax=ax,
    column="Urban_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2001 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Malayalam Urban Population Speaker Distribution Across Kerala Districts (C-16)", fontsize=12
);

In [ ]:
## making a multiples map layout showing each of these
fig, ax = plt.subplots(1, 3, figsize=(20,10))
fig.suptitle("Malayalam Mother Tongue Speaker Distribution by Kerala District (C-16 2011)", fontsize=15);

## using map plotting calls 
kerala_malayalam.plot(
    ax=ax[0],
    column="Total_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=4100000
)

ax[0].set_title("Malayalam Total Population by Kerala District (C-16)");

## using map plotting calls 
kerala_malayalam.plot(
    ax=ax[1],
    column="Rural_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=4100000

)

ax[1].set_title("Malayalam Rural Population by Kerala District (C-16)");


## using map plotting calls 
kerala_malayalam.plot(
    ax=ax[2],
    column="Urban_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=4100000
);

ax[2].set_title("Malayalam Urban Population by Kerala District (C-16)");


## adding a color bar to the bottom 
cmap = cm.viridis

# adding a ScalarMappable object 

plt.Normalize
sm = cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 4100000))
cbar = fig.colorbar(sm, orientation="horizontal", ax=ax, pad=0.05, ticks=None);

## should i put the high concentration and low concentration label?
## cbar.set_ticks([0, 1], labels=['Low Concentration', 'High Concentration'])


In [ ]:
### Now to make it more interactive using plotly and geojson
kerala_malayalam.crs

In [ ]:
# converting geodataframe to geojsontext 
kerala_geojson_text = kerala_malayalam.to_json()
kerala_geojson_text

In [ ]:
# converting to a dictionary
kerala_geojson = json.loads(kerala_geojson_text)

In [ ]:
kerala_geojson

In [ ]:
## plotting map using plotly
interactive_figure = px.choropleth(data_frame=kerala_malayalam, title="Malayalam Total Population by Kerala District (C-16 2011)", geojson=kerala_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Total_P")

In [ ]:
interactive_figure.update_geos(fitbounds="locations", visible=False)
interactive_figure.show()

In [ ]:
## plotting map using plotly (rural)
interactive_rural_figure = px.choropleth(data_frame=kerala_malayalam, title="Malayalam Rural Population by Kerala District (C-16 2011)", geojson=kerala_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Rural_P")

In [ ]:
interactive_rural_figure.update_geos(fitbounds="locations", visible=False)
interactive_rural_figure.show()

In [ ]:
## plotting map using plotly (urban)
interactive_urban_figure = px.choropleth(data_frame=kerala_malayalam, title="Malayalam Urban Population by Kerala District (C-16 2011)", geojson=kerala_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Urban_P")

In [ ]:
interactive_urban_figure.update_geos(fitbounds="locations", visible=False)
interactive_urban_figure.show()

In [ ]:
## Making a multiple map layout for total, urban, and rural populations 

# Tamil Mother Tongue Speakers in Kerala

In [ ]:
## getting all kerala district boundaries
kerala_districts = districts[districts["NAME_1"] == "Kerala"].copy()


# taking all the malayalam rows 
tamil_df = district_kerala_df[district_kerala_df["Mother tongue"] == "Tamil"]

# merging 
kerala_tamil = kerala_districts.merge(
    tamil_df[["District", "Total_P", "Rural_P", "Urban_P"]],
    left_on = "NAME_2",
    right_on = "District", 
    how = "left", 
    validate="one_to_one"
)

In [ ]:
kerala_tamil

In [ ]:
## making a full district map 

fig, ax = plt.subplots(figsize=(8,8))

kerala_tamil.plot(
    ax=ax,
    column="Total_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2001 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Tamil Total Population Speaker Distribution Across Kerala Districts (C-16)", fontsize=12
);

In [ ]:
## making a full district map 

fig, ax = plt.subplots(figsize=(8,8))

kerala_tamil.plot(
    ax=ax,
    column="Rural_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2001 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Tamil Rural Population Speaker Distribution Across Kerala Districts (C-16)", fontsize=12
);

In [ ]:
## making a full district map 

fig, ax = plt.subplots(figsize=(8,8))

kerala_tamil.plot(
    ax=ax,
    column="Urban_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2001 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Tamil Urban Population Speaker Distribution Across Kerala Districts (C-16)", fontsize=12
);

In [ ]:
## making a multiples map layout showing each of these
fig, ax = plt.subplots(1, 3, figsize=(20,10))
fig.suptitle("Tamil Mother Tongue Speaker Distribution by Kerala District (C-16 2011)", fontsize=15);

## using map plotting calls 
kerala_tamil.plot(
    ax=ax[0],
    column="Total_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=200000
)

ax[0].set_title("Tamil Total Population by Kerala District (C-16)");

## using map plotting calls 
kerala_tamil.plot(
    ax=ax[1],
    column="Rural_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=200000

)

ax[1].set_title("Tamil Rural Population by Kerala District (C-16)");


## using map plotting calls 
kerala_tamil.plot(
    ax=ax[2],
    column="Urban_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=200000
);

ax[2].set_title("Tamil Urban Population by Kerala District (C-16)");


## adding a color bar to the bottom 
cmap = cm.viridis

# adding a ScalarMappable object 

plt.Normalize
sm = cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 4100000))
cbar = fig.colorbar(sm, orientation="horizontal", ax=ax, pad=0.05, ticks=None);

## should i put the high concentration and low concentration label?
## cbar.set_ticks([0, 1], labels=['Low Concentration', 'High Concentration'])

In [ ]:
kerala_tamil.crs

In [ ]:
# converting geodataframe to geojsontext 
kerala_geojson_text = kerala_tamil.to_json()
kerala_geojson_text

In [ ]:
# converting to a dictionary
kerala_geojson = json.loads(kerala_geojson_text)

In [ ]:
kerala_geojson

In [ ]:
## plotting map using plotly
interactive_figure = px.choropleth(data_frame=kerala_tamil, title="Tamil Total Population by Kerala District (C-16 2011)", geojson=kerala_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Total_P")

In [ ]:
interactive_figure.update_geos(fitbounds="locations", visible=False)
interactive_figure.show()

In [ ]:
## plotting map using plotly (rural)
interactive_rural_figure = px.choropleth(data_frame=kerala_tamil, title="Tamil Rural Population by Kerala District (C-16 2011)", geojson=kerala_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Rural_P")

In [ ]:
interactive_rural_figure.update_geos(fitbounds="locations", visible=False)
interactive_rural_figure.show()

In [ ]:
## plotting map using plotly (urban)
interactive_urban_figure = px.choropleth(data_frame=kerala_tamil, title="Tamil Urban Population by Kerala District (C-16 2011)", geojson=kerala_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Urban_P")

In [ ]:
interactive_urban_figure.update_geos(fitbounds="locations", visible=False)
interactive_urban_figure.show()

## Telugu Mother Tongue Speakers in Kerala

In [ ]:
## getting all kerala district boundaries
kerala_districts = districts[districts["NAME_1"] == "Kerala"].copy()


# taking all the malayalam rows 
telugu_df = district_kerala_df[district_kerala_df["Mother tongue"] == "Telugu"]

# merging 
kerala_telugu = kerala_districts.merge(
    telugu_df[["District", "Total_P", "Rural_P", "Urban_P"]],
    left_on = "NAME_2",
    right_on = "District", 
    how = "left", 
    validate="one_to_one"
)

In [ ]:
kerala_telugu

In [ ]:
## making a full district map 

fig, ax = plt.subplots(figsize=(8,8))

kerala_telugu.plot(
    ax=ax,
    column="Total_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2001 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Telugu Total Population Speaker Distribution Across Kerala Districts (C-16)", fontsize=12
);

In [ ]:
## making a full district map 

fig, ax = plt.subplots(figsize=(8,8))

kerala_telugu.plot(
    ax=ax,
    column="Rural_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2001 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Telugu Rural Population Speaker Distribution Across Kerala Districts (C-16)", fontsize=12
);

In [ ]:
## making a full district map 

fig, ax = plt.subplots(figsize=(8,8))

kerala_telugu.plot(
    ax=ax,
    column="Urban_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2001 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Telugu Urban Population Speaker Distribution Across Kerala Districts (C-16)", fontsize=12
);

In [ ]:
## making a multiples map layout showing each of these
fig, ax = plt.subplots(1, 3, figsize=(20,10))
fig.suptitle("Telugu Mother Tongue Speaker Distribution by Kerala District (C-16 2011)", fontsize=15);

## using map plotting calls 
kerala_telugu.plot(
    ax=ax[0],
    column="Total_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=20000
)

ax[0].set_title("Telugu Total Population by Kerala District (C-16)");

## using map plotting calls 
kerala_telugu.plot(
    ax=ax[1],
    column="Rural_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=20000

)

ax[1].set_title("Telugu Rural Population by Kerala District (C-16)");


## using map plotting calls 
kerala_telugu.plot(
    ax=ax[2],
    column="Urban_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=20000
);

ax[2].set_title("Telugu Urban Population by Kerala District (C-16)");


## adding a color bar to the bottom 
cmap = cm.viridis

# adding a ScalarMappable object 

plt.Normalize
sm = cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 20000))
cbar = fig.colorbar(sm, orientation="horizontal", ax=ax, pad=0.05, ticks=None);

## should i put the high concentration and low concentration label?
## cbar.set_ticks([0, 1], labels=['Low Concentration', 'High Concentration'])

In [ ]:
kerala_telugu.crs

In [ ]:
# converting geodataframe to geojsontext 
kerala_geojson_text = kerala_telugu.to_json()
kerala_geojson_text

In [ ]:
# converting to a dictionary
kerala_geojson = json.loads(kerala_geojson_text)

In [ ]:
kerala_geojson

In [ ]:
## plotting map using plotly
interactive_figure = px.choropleth(data_frame=kerala_telugu, title="Telugu Total Population by Kerala District (C-16 2011)", geojson=kerala_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Total_P")

In [ ]:
interactive_figure.update_geos(fitbounds="locations", visible=False)
interactive_figure.show()

In [ ]:
## plotting map using plotly (rural)
interactive_rural_figure = px.choropleth(data_frame=kerala_telugu, title="Telugu Rural Population by Kerala District (C-16 2011)", geojson=kerala_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Rural_P")

In [ ]:
interactive_rural_figure.update_geos(fitbounds="locations", visible=False)
interactive_rural_figure.show()

In [ ]:
## plotting map using plotly (urban)
interactive_urban_figure = px.choropleth(data_frame=kerala_telugu, title="Telugu Urban Population by Kerala District (C-16 2011)", geojson=kerala_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Urban_P")

In [ ]:
interactive_urban_figure.update_geos(fitbounds="locations", visible=False)
interactive_urban_figure.show()

# Kannada Mother Tongue Speakers in Kerala 

In [ ]:
## getting all kerala district boundaries
kerala_districts = districts[districts["NAME_1"] == "Kerala"].copy()


# taking all the malayalam rows 
kannada_df = district_kerala_df[district_kerala_df["Mother tongue"] == "Kannada"]

# merging 
kerala_kannada = kerala_districts.merge(
    kannada_df[["District", "Total_P", "Rural_P", "Urban_P"]],
    left_on = "NAME_2",
    right_on = "District", 
    how = "left", 
    validate="one_to_one"
);

In [ ]:
## making sure total_p shows for all districts once merged
kerala_kannada

In [ ]:
## making a full district map 

fig, ax = plt.subplots(figsize=(8,8))

kerala_kannada.plot(
    ax=ax,
    column="Total_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2001 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Kannada Total Population Speaker Distribution Across Kerala Districts (C-16)", fontsize=12
);

In [ ]:
## making a full district map 

fig, ax = plt.subplots(figsize=(8,8))

kerala_kannada.plot(
    ax=ax,
    column="Rural_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2001 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Kannada Rural Population Speaker Distribution Across Kerala Districts (C-16)", fontsize=12
);

In [ ]:
## making a full district map 

fig, ax = plt.subplots(figsize=(8,8))

kerala_kannada.plot(
    ax=ax,
    column="Urban_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2001 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Kannada Urban Population Speaker Distribution Across Kerala Districts (C-16)", fontsize=12
);

In [ ]:
## making a multiples map layout showing each of these
fig, ax = plt.subplots(1, 3, figsize=(20,10))
fig.suptitle("Kannada Mother Tongue Speaker Distribution by Kerala District (C-16 2011)", fontsize=15);

## using map plotting calls 
kerala_kannada.plot(
    ax=ax[0],
    column="Total_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=60000
)

ax[0].set_title("Kannada Total Population by Kerala District (C-16)");

## using map plotting calls 
kerala_kannada.plot(
    ax=ax[1],
    column="Rural_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=60000

)

ax[1].set_title("Kannada Rural Population by Kerala District (C-16)");


## using map plotting calls 
kerala_kannada.plot(
    ax=ax[2],
    column="Urban_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=60000
);

ax[2].set_title("Kannada Urban Population by Kerala District (C-16)");


## adding a color bar to the bottom 
cmap = cm.viridis

# adding a ScalarMappable object 

plt.Normalize
sm = cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 60000))
cbar = fig.colorbar(sm, orientation="horizontal", ax=ax, pad=0.05, ticks=None);

## should i put the high concentration and low concentration label?
## cbar.set_ticks([0, 1], labels=['Low Concentration', 'High Concentration'])

In [ ]:
kerala_kannada.crs

In [ ]:
# converting geodataframe to geojsontext 
kerala_geojson_text = kerala_kannada.to_json()
kerala_geojson_text

In [ ]:
# converting to a dictionary
kerala_geojson = json.loads(kerala_geojson_text)

In [ ]:
kerala_geojson

In [ ]:
## plotting map using plotly
interactive_figure = px.choropleth(data_frame=kerala_kannada, title="Kannada Total Population by Kerala District (C-16 2011)", geojson=kerala_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Total_P")

In [ ]:
interactive_figure.update_geos(fitbounds="locations", visible=False)
interactive_figure.show()

In [ ]:
## plotting map using plotly
interactive_figure = px.choropleth(data_frame=kerala_kannada, title="Kannada Rural Population by Kerala District (C-16 2011)", geojson=kerala_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Rural_P")

In [ ]:
interactive_figure.update_geos(fitbounds="locations", visible=False)
interactive_figure.show()

In [ ]:
## plotting map using plotly
interactive_figure = px.choropleth(data_frame=kerala_kannada, title="Kannada Urban Population by Kerala District (C-16 2011)", geojson=kerala_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Urban_P")

In [ ]:
interactive_figure.update_geos(fitbounds="locations", visible=False)
interactive_figure.show()

## Multiples Map of all 4 Dravidian languages spoken in Kerala

In [ ]:
## making a multiples map layout showing each of these
fig, ax = plt.subplots(4, 3, figsize=(15,22))
fig.suptitle("The main 4 Dravidian Mother Tongue Speaker Distribution by Kerala District (C-16 2011)", fontsize=15);
## fig.tight_layout()
fig.subplots_adjust(wspace=0.08, hspace=0.35)


## using map plotting calls 
kerala_malayalam.plot(
    ax=ax[0, 0],
    column="Total_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=4100000
)

ax[0,0].set_title("Malayalam Total Population");

## using map plotting calls 
kerala_malayalam.plot(
    ax=ax[0, 1],
    column="Rural_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=4100000

)

ax[0,1].set_title("Malayalam Rural Population");


## using map plotting calls 
kerala_malayalam.plot(
    ax=ax[0,2],
    column="Urban_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=4100000
);

ax[0,2].set_title("Malayalam Urban Population");

# Tamil speakers

## using map plotting calls 
kerala_tamil.plot(
    ax=ax[1,0],
    column="Total_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=200000
)

ax[1, 0].set_title("Tamil Total Population");

## using map plotting calls 
kerala_tamil.plot(
    ax=ax[1,1],
    column="Rural_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=200000

)

ax[1, 1].set_title("Tamil Rural Population");


## using map plotting calls 
kerala_tamil.plot(
    ax=ax[1,2],
    column="Urban_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=200000
);

ax[1, 2].set_title("Tamil Urban Population");


## Telugu speakers
## using map plotting calls 
kerala_telugu.plot(
    ax=ax[2,0],
    column="Total_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=20000
)

ax[2,0].set_title("Telugu Total Population");

## using map plotting calls 
kerala_telugu.plot(
    ax=ax[2,1],
    column="Rural_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=20000

)

ax[2,1].set_title("Telugu Rural Population");


## using map plotting calls 
kerala_telugu.plot(
    ax=ax[2,2],
    column="Urban_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=20000
);

ax[2,2].set_title("Telugu Urban Population)");


### Kannada speakers

## using map plotting calls 
kerala_kannada.plot(
    ax=ax[3,0],
    column="Total_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=60000
)

ax[3,0].set_title("Kannada Total Population");

## using map plotting calls 
kerala_kannada.plot(
    ax=ax[3,1],
    column="Rural_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=60000

)

ax[3,1].set_title("Kannada Rural Population");


## using map plotting calls 
kerala_kannada.plot(
    ax=ax[3,2],
    column="Urban_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=60000
);

ax[3,2].set_title("Kannada Urban Population");



## adding a color bar to the bottom 
cmap = cm.viridis

# adding a ScalarMappable object 

plt.Normalize
sm = cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 4100000))
cbar = fig.colorbar(sm, orientation="horizontal", ax=ax, pad=0.05, ticks=None);

## should i put the high concentration and low concentration label?
## cbar.set_ticks([0, 1], labels=['Low Concentration', 'High Concentration'])


***

# Linguistic Distribution of Tamil Nadu

In [ ]:
tamil_nadu_c_16_data = pd.read_csv("dravidian_languages_tamil_nadu_district-level-c-16-2011.csv")
tamil_nadu_c_16_data

In [ ]:
## taking state GeoDataFrame 
tamil_nadu_shape = india_states[india_states["NAME_1"] == "Tamil Nadu"]
tamil_nadu_shape.plot();

In [ ]:
## want to see Tamil Nadu district data 
districts[districts["NAME_1"] == "TamilNadu"]

In [ ]:
chennai = districts[(districts["NAME_1"]=="TamilNadu") & (districts["NAME_2"]=="Chennai")]
chennai

In [ ]:
# plotting chennai district in tamil nadu 
chennai.plot();

In [ ]:
# getting the tamil district boundaries 
tamil_districts = districts[districts["NAME_1"] == "TamilNadu"].copy()

In [ ]:
# taking all the tamil rows 
tamil_df = tamil_nadu_c_16_data[tamil_nadu_c_16_data["Mother tongue"] == "Tamil"]

# merging 
tamil_nadu_tamil = tamil_districts.merge(
    tamil_df[["District", "Total_P", "Rural_P", "Urban_P"]],
    left_on = "NAME_2",
    right_on = "District", 
    how = "left", 
    validate="one_to_one"
)

In [ ]:
## making sure total_p shows for all districts once merged
tamil_nadu_tamil 

In [ ]:
## making a full district map for 

fig, ax = plt.subplots(figsize=(8,8))

tamil_nadu_tamil.plot(
    ax=ax,
    column="Total_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Tamil Total Population Speaker Distribution Across Tamil Nadu Districts (C-16)", fontsize=12
);

In [ ]:
## making a full district map for 

fig, ax = plt.subplots(figsize=(8,8))

tamil_nadu_tamil.plot(
    ax=ax,
    column="Rural_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Tamil Rural Population Speaker Distribution Across Tamil Nadu Districts (C-16)", fontsize=12
);

In [ ]:
## making a full district map for 

fig, ax = plt.subplots(figsize=(8,8))

tamil_nadu_tamil.plot(
    ax=ax,
    column="Urban_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Tamil Urban Population Speaker Distribution Across Tamil Nadu Districts (C-16)", fontsize=12
);

In [ ]:
## making a multiples map layout showing each of these
fig, ax = plt.subplots(1, 3, figsize=(20,10))
fig.suptitle("Tamil Mother Tongue Speaker Distribution by Tamil Nadu District (C-16 2011)", fontsize=15);

## using map plotting calls 
tamil_nadu_tamil.plot(
    ax=ax[0],
    column="Total_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=4100000
)

ax[0].set_title("Tamil Total Population by Tamil Nadu District (C-16)");

## using map plotting calls 
tamil_nadu_tamil.plot(
    ax=ax[1],
    column="Rural_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=4100000

)

ax[1].set_title("Tamil Rural Population by Tamil Nadu District (C-16)");


## using map plotting calls 
tamil_nadu_tamil.plot(
    ax=ax[2],
    column="Urban_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=4100000
);

ax[2].set_title("Tamil Urban Population by Tamil Nadu District (C-16)");


## adding a color bar to the bottom 
cmap = cm.viridis

# adding a ScalarMappable object 

plt.Normalize
sm = cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 4100000))
cbar = fig.colorbar(sm, orientation="horizontal", ax=ax, pad=0.05, ticks=None);

## should i put the high concentration and low concentration label?
## cbar.set_ticks([0, 1], labels=['Low Concentration', 'High Concentration'])

## Malayalam Mother Tongue Speakers in Tamil Nadu 

In [ ]:
## getting all tamil nadu district boundaries
malayalam_tamil_nadu_df = tamil_nadu_c_16_data[tamil_nadu_c_16_data["Mother tongue"] == "Malayalam"]

# merging 
tamil_nadu_malayalam = tamil_districts.merge(
    malayalam_tamil_nadu_df[["District", "Total_P", "Rural_P", "Urban_P"]],
    left_on = "NAME_2",
    right_on = "District", 
    how = "left", 
    validate="one_to_one"
)

In [ ]:
## making sure total_p shows for all districts once merged
tamil_nadu_malayalam 

In [ ]:
## making a full district map for 

fig, ax = plt.subplots(figsize=(8,8))

tamil_nadu_malayalam.plot(
    ax=ax,
    column="Total_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Malayalam Total Population Speaker Distribution Across Tamil Nadu Districts (C-16)", fontsize=12
);

In [ ]:
## making a full district map for 

fig, ax = plt.subplots(figsize=(8,8))

tamil_nadu_malayalam.plot(
    ax=ax,
    column="Rural_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Malayalam Rural Population Speaker Distribution Across Tamil Nadu Districts (C-16)", fontsize=12
);

In [ ]:
## making a full district map for 

fig, ax = plt.subplots(figsize=(8,8))

tamil_nadu_malayalam.plot(
    ax=ax,
    column="Urban_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Malayalam Urban Population Speaker Distribution Across Tamil Nadu Districts (C-16)", fontsize=12
);

In [ ]:
## making a multiples map layout showing each of these
fig, ax = plt.subplots(1, 3, figsize=(20,10))
fig.suptitle("Malayalam Mother Tongue Speaker Distribution by Tamil Nadu District (C-16 2011)", fontsize=15);

## using map plotting calls 
tamil_nadu_malayalam.plot(
    ax=ax[0],
    column="Total_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=170000
)

ax[0].set_title("Malayalam Total Population by Tamil Nadu District (C-16)");

## using map plotting calls 
tamil_nadu_malayalam.plot(
    ax=ax[1],
    column="Rural_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=170000

)

ax[1].set_title("Malayalam Rural Population by Tamil Nadu District (C-16)");


## using map plotting calls 
tamil_nadu_malayalam.plot(
    ax=ax[2],
    column="Urban_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=170000
);

ax[2].set_title("Malayalam Urban Population by Tamil Nadu District (C-16)");


## adding a color bar to the bottom 
cmap = cm.viridis

# adding a ScalarMappable object 

plt.Normalize
sm = cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 4100000))
cbar = fig.colorbar(sm, orientation="horizontal", ax=ax, pad=0.05, ticks=None);

## should i put the high concentration and low concentration label?
## cbar.set_ticks([0, 1], labels=['Low Concentration', 'High Concentration'])

## Telugu Mother Tongue Speakers in Tamil Nadu

In [ ]:
## getting all tamil nadu district boundaries
telugu_tamil_nadu_df = tamil_nadu_c_16_data[tamil_nadu_c_16_data["Mother tongue"] == "Telugu"]

# merging 
tamil_nadu_telugu = tamil_districts.merge(
    telugu_tamil_nadu_df[["District", "Total_P", "Rural_P", "Urban_P"]],
    left_on = "NAME_2",
    right_on = "District", 
    how = "left", 
    validate="one_to_one"
)

In [ ]:
tamil_nadu_telugu

In [ ]:
# making district map for telugu 
fig, ax = plt.subplots(figsize=(8,8))

tamil_nadu_telugu.plot(
    ax=ax,
    column="Total_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Telugu Total Population Speaker Distribution Across Tamil Nadu Districts (C-16)", fontsize=12
);

In [ ]:
# making district map for telugu 
fig, ax = plt.subplots(figsize=(8,8))

tamil_nadu_telugu.plot(
    ax=ax,
    column="Rural_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Telugu Rural Population Speaker Distribution Across Tamil Nadu Districts (C-16)", fontsize=12
);

In [ ]:
# making district map for telugu 
fig, ax = plt.subplots(figsize=(8,8))

tamil_nadu_telugu.plot(
    ax=ax,
    column="Urban_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Telugu Urban Population Speaker Distribution Across Tamil Nadu Districts (C-16)", fontsize=12
);

In [ ]:
## making a multiples map layout showing each of these
fig, ax = plt.subplots(1, 3, figsize=(20,10))
fig.suptitle("Telugu Mother Tongue Speaker Distribution by Tamil Nadu District (C-16 2011)", fontsize=15);

## using map plotting calls 
tamil_nadu_telugu.plot(
    ax=ax[0],
    column="Total_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=600000
)

ax[0].set_title("Telugu Total Population by Tamil Nadu District (C-16)");

## using map plotting calls 
tamil_nadu_telugu.plot(
    ax=ax[1],
    column="Rural_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=600000

)

ax[1].set_title("Telugu Rural Population by Tamil Nadu District (C-16)");


## using map plotting calls 
tamil_nadu_telugu.plot(
    ax=ax[2],
    column="Urban_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=600000
);

ax[2].set_title("Telugu Urban Population by Tamil Nadu District (C-16)");


## adding a color bar to the bottom 
cmap = cm.viridis

# adding a ScalarMappable object 

plt.Normalize
sm = cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 600000))
cbar = fig.colorbar(sm, orientation="horizontal", ax=ax, pad=0.05, ticks=None);

## should i put the high concentration and low concentration label?
## cbar.set_ticks([0, 1], labels=['Low Concentration', 'High Concentration'])

In [ ]:
tamil_nadu_telugu.crs

In [ ]:
telugu_geojson_text = tamil_nadu_telugu.to_json()
telugu_geojson_text

In [ ]:
# converting to a dictionary
telugu_geojson = json.loads(telugu_geojson_text)

In [ ]:
telugu_geojson

In [ ]:
## plotting map using plotly
interactive_figure = px.choropleth(data_frame=tamil_nadu_telugu, title="Telugu Total Population by Tamil Nadu District (C-16 2011)", geojson=telugu_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Total_P")

In [ ]:
interactive_figure.update_geos(fitbounds="locations", visible=False)
interactive_figure.show()

In [ ]:
## plotting map using plotly
interactive_figure = px.choropleth(data_frame=tamil_nadu_telugu, title="Telugu Rural Population by Tamil Nadu District (C-16 2011)", geojson=telugu_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Rural_P")

In [ ]:
interactive_figure.update_geos(fitbounds="locations", visible=False)
interactive_figure.show()

In [ ]:
## plotting map using plotly
interactive_figure = px.choropleth(data_frame=tamil_nadu_telugu, title="Telugu Urban Population by Tamil Nadu District (C-16 2011)", geojson=telugu_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Urban_P")

In [ ]:
interactive_figure.update_geos(fitbounds="locations", visible=False)
interactive_figure.show()

# Kannada Mother Tongue Speakers in Tamil Nadu 

In [ ]:
## getting all tamil nadu district boundaries
kannada_tamil_nadu_df = tamil_nadu_c_16_data[tamil_nadu_c_16_data["Mother tongue"] == "Kannada"]

# merging 
tamil_nadu_kannada = tamil_districts.merge(
    kannada_tamil_nadu_df[["District", "Total_P", "Rural_P", "Urban_P"]],
    left_on = "NAME_2",
    right_on = "District", 
    how = "left", 
    validate="one_to_one"
)

In [ ]:
tamil_nadu_kannada

In [ ]:
# making district map for telugu 
fig, ax = plt.subplots(figsize=(8,8))

tamil_nadu_kannada.plot(
    ax=ax,
    column="Total_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Kannada Total Population Speaker Distribution Across Tamil Nadu Districts (C-16)", fontsize=12
);

In [ ]:
# making district map for telugu 
fig, ax = plt.subplots(figsize=(8,8))

tamil_nadu_kannada.plot(
    ax=ax,
    column="Rural_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Kannada Rural Population Speaker Distribution Across Tamil Nadu Districts (C-16)", fontsize=12
);

In [ ]:
# making district map for telugu 
fig, ax = plt.subplots(figsize=(8,8))

tamil_nadu_kannada.plot(
    ax=ax,
    column="Urban_P",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.5,
    legend_kwds={"label": "Speaker Population (2011 Census)", "orientation": "horizontal"}

);

ax.set_title(
    "2011 Urban Population Speaker Distribution Across Tamil Nadu Districts (C-16)", fontsize=12
);

In [ ]:
## making a multiples map layout showing each of these
fig, ax = plt.subplots(1, 3, figsize=(20,10))
fig.suptitle("Kannada Mother Tongue Speaker Distribution by Tamil Nadu District (C-16 2011)", fontsize=15);

## using map plotting calls 
tamil_nadu_kannada.plot(
    ax=ax[0],
    column="Total_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=600000
)

ax[0].set_title("Kannada Total Population by Tamil Nadu District (C-16)");

## using map plotting calls 
tamil_nadu_kannada.plot(
    ax=ax[1],
    column="Rural_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=600000

)

ax[1].set_title("Kannada Rural Population by Tamil Nadu District (C-16)");


## using map plotting calls 
tamil_nadu_kannada.plot(
    ax=ax[2],
    column="Urban_P",
    edgecolor="white",
    linewidth=0.5,
    cmap="viridis",
    vmin=0,
    vmax=600000
);

ax[2].set_title("Kannada Urban Population by Tamil Nadu District (C-16)");


## adding a color bar to the bottom 
cmap = cm.viridis

# adding a ScalarMappable object 

plt.Normalize
sm = cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 600000))
cbar = fig.colorbar(sm, orientation="horizontal", ax=ax, pad=0.05, ticks=None);

## should i put the high concentration and low concentration label?
## cbar.set_ticks([0, 1], labels=['Low Concentration', 'High Concentration'])

In [ ]:
tamil_nadu_kannada.crs

In [ ]:
# converting geodataframe to geojsontext 
tamil_nadu_geojson_text = tamil_nadu_kannada.to_json()
tamil_nadu_geojson_text

In [ ]:
# converting to a dictionary
tamil_nadu_geojson = json.loads(tamil_nadu_geojson_text)

In [ ]:
tamil_nadu_geojson

In [ ]:
## plotting map using plotly
interactive_figure = px.choropleth(data_frame=tamil_nadu_kannada, title="Kannada Total Population by Tamil Nadu District (C-16 2011)", geojson=tamil_nadu_geojson, locations="NAME_2", featureidkey="properties.NAME_2", color="Total_P")

In [ ]:
interactive_figure.update_geos(fitbounds="locations", visible=False)
interactive_figure.show()

In [ ]:
## plotting map using plotly
interactive_figure = px.choropleth(data_frame=tamil_nadu_kannada, title="Kannada Rural Population by Kerala District (C-16 2011)", geojson=tamil_nadu_kannada, locations="NAME_2", featureidkey="properties.NAME_2", color="Rural_P")

In [ ]:
interactive_figure.update_geos(fitbounds="locations", visible=False)
interactive_figure.show()

In [ ]:
## plotting map using plotly
interactive_figure = px.choropleth(data_frame=tamil_nadu_kannada, title="Kannada Urban Population by Kerala District (C-16 2011)", geojson=tamil_nadu_kannada, locations="NAME_2", featureidkey="properties.NAME_2", color="Urban_P")

In [ ]:
interactive_figure.update_geos(fitbounds="locations", visible=False)
interactive_figure.show()